# Fake News Detection - Phase 1: Data Preprocessing

This notebook demonstrates the preprocessing and cleaning steps of the Fake News Detection pipeline. We will:
1. Load the raw datasets (`True.csv` and `Fake.csv`).
2. Explore distributions (counts, subjects, and text lengths).
3. Perform NLP cleaning (case folding, URL removal, tokenization, stopword removal, and lemmatization).
4. Save the processed data for feature extraction.

In [ ]:
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add src to path to import utility clean_text
sys.path.append(os.path.abspath('../src'))
from utils import clean_text

## 1. Load the Datasets

In [ ]:
# Ensure datasets exist; if not, generate mock data
if not (os.path.exists('../dataset/True.csv') and os.path.exists('../dataset/Fake.csv')):
    print("Mock datasets not found. Generating mock datasets...")
    from generate_mock_data import generate_news_dataset
    generate_news_dataset(1000)

df_true = pd.read_csv('../dataset/True.csv')
df_fake = pd.read_csv('../dataset/Fake.csv')

print(f"True news shape: {df_true.shape}")
print(f"Fake news shape: {df_fake.shape}")

Let's see some sample articles.

In [ ]:
print("--- True News Sample ---")
print(df_true.head(2)[['title', 'subject']])
print("\n--- Fake News Sample ---")
print(df_fake.head(2)[['title', 'subject']])

## 2. Basic EDA (Exploratory Data Analysis)

In [ ]:
# Add class labels
df_true['label'] = 'True'
df_fake['label'] = 'Fake'

# Merge datasets
df = pd.concat([df_true, df_fake], ignore_index=True)

# Plot distribution of news categories
plt.figure(figsize=(10, 5))
sns.countplot(data=df, x='subject', hue='label', palette='viridis')
plt.title('Article Counts by Subject and Authenticity')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

Let's inspect the word counts of the text in true vs. fake news articles.

In [ ]:
df['char_length'] = df['text'].apply(lambda x: len(str(x)))
df['word_count'] = df['text'].apply(lambda x: len(str(x).split()))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(data=df, x='word_count', hue='label', kde=True, bins=30, ax=axes[0], palette='magma')
axes[0].set_title('Word Count Distribution')
axes[0].set_xlabel('Words per Article')

sns.boxplot(data=df, y='word_count', x='label', ax=axes[1], palette='magma')
axes[1].set_title('Word Count Outliers')
axes[1].set_ylabel('Words per Article')

plt.tight_layout()
plt.show()

## 3. Cleaning the Text Data

In [ ]:
# Combine Title and Text body
df['full_content'] = df['title'] + " " + df['text']

# Clean sample article
sample_text = df['full_content'].iloc[0]
cleaned_sample = clean_text(sample_text)

print("=== ORIGINAL TEXT ===")
print(sample_text[:300] + "...")
print("\n=== CLEANED TEXT ===")
print(cleaned_sample[:300] + "...")

Let's clean the entire dataset content. This might take a few moments.

In [ ]:
print("Cleaning full dataset...")
df['cleaned_content'] = df['full_content'].apply(clean_text)
df['label_binary'] = df['label'].map({'True': 1, 'Fake': 0})

# Save preprocessed dataset
os.makedirs('../dataset', exist_ok=True)
df_output = df[['title', 'text', 'subject', 'date', 'label_binary', 'cleaned_content']]
df_output.rename(columns={'label_binary': 'label'}, inplace=True)
df_output.to_csv('../dataset/processed_data.csv', index=False)
print("Saved preprocessed data to dataset/processed_data.csv")